<div align="center">
  <a href="https://colab.research.google.com/github/ActionPace/ensemble-divergence/blob/main/ensemble_divergence_demo.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

# Ensemble Divergence — Structural Opacity Demo
### *Toward a Control Architecture for Composable AI Execution Systems*
**David E. King · ActionPace · April 2026**

---

This notebook demonstrates **ensemble divergence as a structural signal** for identifying
implicit contracts in source code — obligations that components place on their callers
without declaring them in any interface.

The ensemble pairs **Apertus** (Swiss AI, Llama-family, trained at CSCS) with **Mistral**
(French, independent architecture) — a deliberately heterogeneous combination.
If both independently locate the same orphaned invariant, the finding is harder
to dismiss as a model-specific artifact than if only one model found it.

### Notebook structure

| Section | Runtime | What it shows |
|---------|---------|---------------|
| **Section 1** | T4 free tier | Apertus 8B vs Mistral 7B — core methodology, anyone can run |
| **Section 2** | A100 (any paid plan) | Apertus 70B vs Mistral 7B — depth-of-read differential |
| **Section 3** | A100 (any paid plan) | Apertus 70B vs Mistral Nemo 12B — full architecture differential |

Sections 2 and 3 skip gracefully on free-tier runtimes.
**Section 1 is complete and self-contained.**

---
**Repository:** [github.com/ActionPace/ensemble-divergence](https://github.com/ActionPace/ensemble-divergence)  
**Paper:** arXiv forthcoming · cs.SE / cs.AI

---
## Section 0 — Hardware Detection
Run this first. Detects your runtime and sets flags used by all sections below.

In [1]:
#@title Detect hardware and set runtime flags
import subprocess, shutil, psutil, os

machine_type        = "CPU"
total_gpu_memory_gb = 0.0
total_ram_gb        = psutil.virtual_memory().total / 1024**3
_, _, free_disk     = shutil.disk_usage("/")
free_disk_gb        = free_disk / 1024**3

try:
    gpu_name = subprocess.check_output(
        "nvidia-smi --query-gpu=gpu_name --format=csv,noheader",
        shell=True, text=True).strip()
    smi_out = subprocess.check_output(
        "nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits",
        shell=True, text=True).strip()
    total_gpu_memory_gb = int(smi_out.splitlines()[0]) / 1024
    if   "T4"   in gpu_name: machine_type = "T4"
    elif "L4"   in gpu_name: machine_type = "L4"
    elif "A100" in gpu_name: machine_type = "A100"
    else:                    machine_type = "GPU_Other"
except Exception:
    gpu_name = "None"

is_a100       = machine_type == "A100"
is_high_ram = total_gpu_memory_gb >= 75

can_run_s1 = True
can_run_s2 = is_a100
can_run_s3 = is_a100 and is_high_ram

# ── A100 partial-layer config ──────────────────────────────────────────────────
# 70B Q5_K_S is ~47 GB. Standard A100 = 40 GB VRAM.
# Set to number of transformer layers to keep on GPU; rest offload to CPU RAM.
# Start at 60 and reduce by 4 until it loads without OOM.
# Leave as -1 if you have 80GB A100 or High-RAM runtime enabled.
A100_70B_PARTIAL_LAYERS = 72   # <- fill in after testing

if is_a100 and is_high_ram:
    layers_70b = -1
elif A100_70B_PARTIAL_LAYERS != -1:
    layers_70b = A100_70B_PARTIAL_LAYERS
else:
    layers_70b = -1

print("=" * 60)
print(" RUNTIME SUMMARY")
print("=" * 60)
print(f"  GPU:          {gpu_name}")
print(f"  GPU VRAM:     {total_gpu_memory_gb:.1f} GiB")
print(f"  System RAM:   {total_ram_gb:.1f} GiB")
print(f"  Free disk:    {free_disk_gb:.1f} GiB")
print()
print("=" * 60)
print(" SECTION AVAILABILITY")
print("=" * 60)
print(f"  Section 1 (Apertus 8B vs Mistral 7B):        {'YES' if can_run_s1 else 'needs GPU'}")
print(f"  Section 2 (Apertus 70B vs Mistral 7B):       {'YES' if can_run_s2 else 'needs A100 — any paid Colab plan'}")
print(f"  Section 3 (Apertus 70B vs Mistral Nemo 12B): {'YES' if can_run_s2 else 'needs A100 + High-RAM or 80GB A100'}")
if is_a100 and not is_high_ram and A100_70B_PARTIAL_LAYERS == -1:
    print()
    print("  NOTE: A100 40GB detected. Section 2 will attempt full 70B load.")
    print("  If OOM: set A100_70B_PARTIAL_LAYERS above (start at 60, step -4)")
    print("  Or: Runtime > Change runtime type > High-RAM (Pro only)")
print("=" * 60)

 RUNTIME SUMMARY
  GPU:          NVIDIA A100-SXM4-80GB
  GPU VRAM:     80.0 GiB
  System RAM:   167.1 GiB
  Free disk:    192.6 GiB

 SECTION AVAILABILITY
  Section 1 (Apertus 8B vs Mistral 7B):        YES
  Section 2 (Apertus 70B vs Mistral 7B):       YES
  Section 3 (Apertus 70B vs Mistral Nemo 12B): YES


---
## Section 1 — Free Demo: Apertus 8B vs Mistral 7B
**Runs on T4 free tier. No credentials required.**

Two 8B-class models from different countries and architectures analyze the same file.
Where they agree: high-confidence structural finding.
Where they disagree: a composition seam.

**Target:** `libs/langgraph/langgraph/pregel/main.py` — core execution loop of LangGraph,
one of the six frameworks surveyed in the paper. Identified as high-opacity with
load-bearing implicit contracts in the original survey.

In [2]:
#@title 1.1 Install dependencies and download llama.cpp
from huggingface_hub import hf_hub_download

llama_file = f"llama.cpp-{machine_type}.tar.gz"
print(f"Downloading llama.cpp for {machine_type}...")
hf_hub_download(repo_id="actionpace/Llama-Cpp-Compiles",
                filename=llama_file, local_dir="/content")
if os.path.isdir("/content/llama.cpp"):
    os.system("rm -rf /content/llama.cpp")
os.system(f"tar xf /content/{llama_file} -C /content/")
print("llama.cpp ready")

llama.cpp-A100.tar.gz:   0%|          | 0.00/338M [00:00<?, ?B/s]

llama.cpp ready


In [3]:
#@title 1.2 Download Apertus 8B and Mistral 7B
APERTUS_8B_FILE = "Apertus-8B-Instruct-2509-Q5_K_M.gguf"
APERTUS_8B_REPO = "unsloth/Apertus-8B-Instruct-2509-GGUF"

MISTRAL_7B_FILE = "Mistral-7B-Instruct-v0.3-Q5_K_M.gguf"
MISTRAL_7B_REPO = "bartowski/Mistral-7B-Instruct-v0.3-GGUF"

for repo, fname in [(APERTUS_8B_REPO, APERTUS_8B_FILE),
                    (MISTRAL_7B_REPO,  MISTRAL_7B_FILE)]:
    if not os.path.exists(f"/content/{fname}"):
        print(f"Downloading {fname}...")
        hf_hub_download(repo_id=repo, filename=fname, local_dir="/content")
    print(f"  {fname}")

Apertus-8B-Instruct-2509-Q5_K_M.gguf:   0%|          | 0.00/5.81G [00:00<?, ?B/s]

  Apertus-8B-Instruct-2509-Q5_K_M.gguf


Mistral-7B-Instruct-v0.3-Q5_K_M.gguf:   0%|          | 0.00/5.14G [00:00<?, ?B/s]

  Mistral-7B-Instruct-v0.3-Q5_K_M.gguf


In [4]:
#@title 1.3 Clone LangGraph target file
from pathlib import Path

TARGET_PATH = "libs/langgraph/langgraph/pregel"
TARGET_FILE = "main.py"
clone_dir   = Path("/content/langgraph")

if not clone_dir.exists():
    subprocess.run(["git","clone","--filter=blob:none","--sparse",
                    "--depth=50","https://github.com/langchain-ai/langgraph",
                    str(clone_dir)], capture_output=True)
    subprocess.run(["git","sparse-checkout","set",TARGET_PATH],
                   cwd=clone_dir, capture_output=True)

target_file_path = clone_dir / TARGET_PATH / TARGET_FILE
total_lines = sum(1 for _ in open(target_file_path, encoding="utf-8", errors="ignore"))
print(f"{TARGET_PATH}/{TARGET_FILE}  ({total_lines:,} lines)")

libs/langgraph/langgraph/pregel/main.py  (4,157 lines)


In [5]:
#@title 1.4 Define prompts, helpers, and corpus
import json, time
from openai import OpenAI
import json as _json

OPACITY_SYSTEM = """You are a structural code analyst rating file opacity.

OPACITY = the gap between what a file's interface claims and what it
actually requires from callers. This includes hidden assumptions that
type hints and docstrings do NOT capture.

Score each dimension 0-20. NOTE: A file with good type hints can STILL
score high on dimensions 1-3 if callers must know things beyond what
the types declare.

1. UNDECLARED_PRECONDITIONS (0-20)
   What must callers know BEYOND the type signatures?
   Examples of non-zero scores:
   - Caller must initialize some external resource before calling (score: 8-12)
   - Method assumes a specific ordering of prior calls (score: 10-15)
   - Arguments must satisfy constraints not expressed in types (score: 8-15)
   Score 0 ONLY if type signatures alone are sufficient to use the file correctly.

2. GLOBAL_STATE_RISK (0-20)
   Does this file read or write state that callers cannot see?
   Examples of non-zero scores:
   - Reads from module-level variables or singletons (score: 8-12)
   - Modifies shared caches or registries as a side effect (score: 10-16)
   - Behavior changes based on previous calls to other functions (score: 8-14)
   Score 0 ONLY if every dependency is passed explicitly as an argument.

3. IMPLICIT_COUPLING (0-20)
   Does this file assume things about other parts of the system?
   Examples of non-zero scores:
   - Assumes specific keys exist in config dicts passed by callers (score: 8-14)
   - Relies on other modules being initialized in a specific way (score: 10-16)
   - Behavior depends on environment variables or runtime context (score: 8-15)
   Score 0 ONLY if this file makes zero assumptions about its environment.

4. CONTRACT_COMPLETENESS (0-20, inverted — 0=fully declared, 20=nothing declared)
   How incomplete are the declared contracts overall?

5. STRUCTURAL_CENTRALITY (0-20)
   How much system correctness depends on this file invisibly?

Return ONLY valid JSON, no preamble, no fences:
{"undeclared_preconditions":<int>,"global_state_risk":<int>,"implicit_coupling":<int>,
"contract_completeness":<int>,"structural_centrality":<int>,"total":<int>,
"opacity_score":<float 0-1>,"primary_signal":"<one sentence>",
"load_bearing_functions":["fn1","fn2","fn3"],"summary":"<2 sentences>"}"""

CALLER_BURDEN_SYSTEM = """Identify what callers must know that is not written anywhere.
Return ONLY valid JSON, no preamble, no fences:
{"caller_must_know":[{"item":"<what>","stated_in_interface":<bool>},...],
"riskiest_assumption":"<silent failure mode>","wrapper_strategy":"<one sentence>"}"""


def extract_corpus(path, max_lines=300):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    if len(lines) <= max_lines:
        return "".join(lines), len(lines)
    header = lines[:150]
    kws = ["if ","for ","while ","try:","except","def ","class ",
           "return ","raise ","async ","await ","yield "]
    best, bd = 150, 0
    for i in range(150, max(151, len(lines)-50)):
        d = sum(1 for l in lines[i:i+50] if any(k in l for k in kws))
        if d > bd: bd, best = d, i
    focal = lines[max(0,best-10):best+140]
    return (f"// HEADER ({len(header)} lines)\n" + "".join(header)
            + f"\n// FOCAL (line ~{best})\n" + "".join(focal))[:12000], len(lines)


def parse_json(raw):
    """Robustly extract JSON from model output.
    Handles: markdown fences, prose preamble, truncated output,
    opacity_score returned as 0-100 instead of 0-1.
    Returns empty dict on total failure rather than raising.
    """
    if not raw or not raw.strip():
        print("  [parse_json] Empty response")
        return {}
    raw = raw.strip()

    # Strip markdown fences if present
    if "```" in raw:
        for p in raw.split("```"):
            p = p.strip().lstrip("json").strip()
            if p.startswith("{"): raw = p; break

    # Find the first { anywhere in the response (handles prose preamble)
    first_brace = raw.find("{")
    if first_brace > 0:
        raw = raw[first_brace:]

    # Trim to last closing brace (handles truncated trailing text)
    last_brace = raw.rfind("}")
    if last_brace != -1:
        raw = raw[:last_brace + 1]

    # Fix Python literal booleans/None — 8B models often output these
    # instead of valid JSON true/false/null
    import re
    raw = re.sub(r'\bFalse\b', 'false', raw)
    raw = re.sub(r'\bTrue\b',  'true',  raw)
    raw = re.sub(r'\bNone\b',  'null',  raw)

    try:
        r = json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"  [parse_json] JSONDecodeError: {e}")
        print(f"  [parse_json] Raw (first 200 chars): {raw[:200]}")
        return {}

    # Normalize opacity_score if model returned 0-100 instead of 0-1
    if r.get("opacity_score", 0) > 1.0:
        r["opacity_score"] = r["opacity_score"] / 100.0
    return r

def start_server(model_file, n_gpu_layers=-1, wait=15):
    os.system("pkill -f llama-server 2>/dev/null")
    time.sleep(3)
    cmd = (f"/content/llama.cpp/build/bin/llama-server "
           f"-m /content/{model_file} -c 4096 -ngl {n_gpu_layers} "
           f"--port 12345 --log-file /content/server.log "
           f"--host 0.0.0.0 --jinja")
    subprocess.Popen(f"nohup {cmd} &", shell=True)
    print(f"  Loading {model_file} ({wait}s)...")
    time.sleep(wait)
    try:
        import urllib.request
        urllib.request.urlopen("http://localhost:12345/health", timeout=5)
        print("  Server ready")
    except Exception:
        print("  Still loading — proceeding")


client = OpenAI(api_key="dummy", base_url="http://localhost:12345/v1")

json_schema = """{
  "type": "object",
  "properties": {
    "undeclared_preconditions": {"type": "integer", "minimum": 0, "maximum": 20},
    "global_state_risk":        {"type": "integer", "minimum": 0, "maximum": 20},
    "implicit_coupling":        {"type": "integer", "minimum": 0, "maximum": 20},
    "contract_completeness":    {"type": "integer", "minimum": 0, "maximum": 20},
    "structural_centrality":    {"type": "integer", "minimum": 0, "maximum": 20},
    "total":                    {"type": "integer", "minimum": 0, "maximum": 100},
    "opacity_score":            {"type": "number",  "minimum": 0, "maximum": 1},
    "primary_signal":           {"type": "string"},
    "load_bearing_functions":   {"type": "array", "items": {"type": "string"}},
    "summary":                  {"type": "string"}
  },
  "required": ["undeclared_preconditions","global_state_risk","implicit_coupling",
               "contract_completeness","structural_centrality","total",
               "opacity_score","primary_signal","load_bearing_functions","summary"]
}"""
json_schema_dict = _json.loads(json_schema)

def run_prompt(system, label, max_tokens=800, retries=2):
    """Run a prompt with automatic retry on empty/invalid JSON.
    On retry: increases max_tokens and raises temperature slightly
    to encourage the model to complete its response.
    Returns empty dict if all retries fail.
    """
    for attempt in range(retries + 1):
        if attempt == 0:
            print(f"  Running {label}...")
        else:
            print(f"  Retrying {label} (attempt {attempt+1}, tokens={max_tokens+256*attempt})...")
        try:
            resp = client.chat.completions.create(
                model="local-model",
                max_tokens=max_tokens + 256 * attempt,
                temperature=0.1 + 0.05 * attempt,
                response_format={
                      "type": "json_schema",
                      "json_schema": {
                          "name": "opacity_result",
                          "strict": True,
                          "schema": json_schema_dict
                      }
                  },
                messages=[{"role": "system", "content": system},
                           {"role": "user",   "content": user_prompt}]
            )
            raw = resp.choices[0].message.content or ""
            result = parse_json(raw)
            print(result)
            if result:  # non-empty dict = success
                return result
            print(f"  Empty result on attempt {attempt+1}")
        except Exception as e:
            print(f"  API error on attempt {attempt+1}: {e}")
    print(f"  All retries failed for {label} — returning empty result")
    return {}


def print_result(r, label):
    if not r:
        print(f"  No result to display for {label} — model returned unparseable output")
        return
    t = r.get("total", 0)
    # Recalculate total from dimensions if model returned 0 total but non-zero dims
    if t == 0:
        dims_sum = sum(r.get(k, 0) for k in [
            "undeclared_preconditions", "global_state_risk",
            "implicit_coupling", "contract_completeness", "structural_centrality"
        ])
        if dims_sum > 0: t = dims_sum
    bar = "X"*int(t*32//100) + "."*(32-int(t*32//100))
    tier = "CRITICAL" if t>=80 else "HIGH" if t>=60 else "MEDIUM" if t>=40 else "LOW"
    print(f"\n{'='*65}")
    print(f" OPACITY — {label}")
    print(f" File: {TARGET_PATH}/{TARGET_FILE}")
    print(f"{'='*65}")
    print(f"  [{bar}] {t}/100  {tier}")
    for k,l in [("undeclared_preconditions","Undeclared preconditions"),
                ("global_state_risk",       "Global state risk       "),
                ("implicit_coupling",       "Implicit coupling       "),
                ("contract_completeness",   "Contract completeness   "),
                ("structural_centrality",   "Structural centrality   ")]:
        v = r.get(k,0)
        print(f"  {l}  {v:>2}  {'X'*v + '.'*(20-v)}")
    print(f"  Primary signal: {r.get('primary_signal','')}")
    print(f"  Load-bearing:   {', '.join(r.get('load_bearing_functions',[]))}")
    print(f"  Summary: {r.get('summary','')}")


def print_comparison(oA, oB, bA, bB, lA, lB):
    if not oA or not oB:
        print(f"  Cannot compare — missing results for {'' if oA else lA}{'' if oB else ' / '+lB}")
        return
    W = 72
    def conv(a,b,t=10): return "CONVERGE" if abs(a-b)<=t else "DIVERGE"
    def dlt(a,b): d=b-a; return f"+{d}" if d>0 else str(d) if d<0 else "0"
    tA = oA.get("total", 0)
    tB = oB.get("total", 0)
    sA = tA / 100.0
    sB = tB / 100.0
    print(f"\n{'='*W}")
    print(f" ENSEMBLE COMPARISON: {lA} vs {lB}")
    print(f"{'='*W}")
    for s,l in [(sA,lA),(sB,lB)]:
        bar = "X"*int(s*32)+"."*(32-int(s*32))
        print(f"  {l:<22} [{bar}] {int(s*100):>3}/100")
    print(f"  Spread: {abs(int(sA*100)-int(sB*100))} pts  Overall: {conv(int(sA*100),int(sB*100),15)}")
    print(f"\n{'-'*W}  DIMENSIONS")
    print(f"  {'Dimension':<26} {lA[:10]:>10}  {lB[:10]:>10}  Delta  Signal")
    for k,l in [("undeclared_preconditions","Undeclared preconditions"),
                ("global_state_risk",       "Global state risk       "),
                ("implicit_coupling",       "Implicit coupling       "),
                ("contract_completeness",   "Contract completeness   "),
                ("structural_centrality",   "Structural centrality   ")]:
        vA,vB = oA.get(k,0),oB.get(k,0)
        print(f"  {l:<26} {vA:>10}  {vB:>10}  {dlt(vA,vB):>5}  {conv(vA,vB)}")
    psA = oA.get("primary_signal","")
    psB = oB.get("primary_signal","")
    print(f"\n{'-'*W}  PRIMARY SIGNAL")
    print(f"  {lA}: {psA}")
    print(f"  {lB}: {psB}")
    both_sg = "subgraph" in psA.lower() and "subgraph" in psB.lower()
    if both_sg:
        print("\n  CONVERGENCE: Both models name get_subgraphs")
        print("  Orphaned invariant confirmed at 2x confidence")
        print("  Same structural fact. Two training lineages. Same finding.")
    else:
        print("\n  DIVERGENCE on primary signal — composition seam located")
    fnsA = set(f.lower() for f in oA.get("load_bearing_functions",[]))
    fnsB = set(f.lower() for f in oB.get("load_bearing_functions",[]))
    conv_fns = fnsA & fnsB
    print(f"\n{'-'*W}  LOAD-BEARING FUNCTIONS")
    if conv_fns:
        print(f"  Convergent (both models): {', '.join(sorted(conv_fns))}  [2x confirmed]")
    if fnsA-fnsB: print(f"  {lA} only: {', '.join(sorted(fnsA-fnsB))}")
    if fnsB-fnsA: print(f"  {lB} only: {', '.join(sorted(fnsB-fnsA))}")
    print(f"\n{'-'*W}  CALLER BURDEN")
    for b,l in [(bA,lA),(bB,lB)]:
        items = b.get("caller_must_know",[])
        stated = all(i.get("stated_in_interface",False) for i in items)
        print(f"  {l}: {'ALL STATED' if stated else 'UNSTATED items found'}")
        for item in items:
            m = "stated" if item.get("stated_in_interface") else "UNSTATED"
            print(f"    [{m}] {item.get('item','')[:65]}")
    print(f"\n{'='*W}")
    print("  Agreement establishes confidence.")
    print("  Disagreement locates structure.")
    print("  The most valuable signal is where a model finds what another read past.")
    print("  That is where the orphaned invariants live.")
    print(f"{'='*W}")


corpus, total_lines = extract_corpus(target_file_path)
try:
    git_date = subprocess.check_output(
        ["git","log","-1","--format=%ai","--",str(target_file_path)],
        cwd=clone_dir, text=True, timeout=10).strip()
except Exception:
    git_date = "unknown"

user_prompt = (
    f"File: {TARGET_PATH}/{TARGET_FILE}\nLanguage: Python\n"
    f"Total lines: {total_lines}\nLast commit: {git_date}\n"
    f"Repository: LangGraph (AI agent framework)\n\n"
    f"```python\n{corpus}\n```"
)
print(f"Corpus: {len(corpus):,} chars from {total_lines:,}-line file")
print("Prompts and helpers ready")

Corpus: 9,636 chars from 4,157-line file
Prompts and helpers ready


In [6]:
#@title 1.5 Run Apertus 8B
print("--- Apertus 8B ---")
start_server(APERTUS_8B_FILE, n_gpu_layers=-1, wait=30)
result_apertus_8b = run_prompt(OPACITY_SYSTEM, "Apertus 8B opacity")
burden_apertus_8b = run_prompt(CALLER_BURDEN_SYSTEM, "Apertus 8B caller burden", 600)
print_result(result_apertus_8b, "Apertus 8B")

--- Apertus 8B ---
  Loading Apertus-8B-Instruct-2509-Q5_K_M.gguf (30s)...
  Server ready
  Running Apertus 8B opacity...
{'undeclared_preconditions': 12, 'global_state_risk': 16, 'implicit_coupling': 14, 'contract_completeness': 12, 'structural_centrality': 18, 'total': 78, 'opacity_score': 0.96, 'primary_signal': 'This file has a high opacity score due to its reliance on global state, implicit coupling, and incomplete contracts. The file assumes specific keys exist in config dicts, relies on other modules being initialized in a specific way, and modifies shared caches as a side effect.', 'load_bearing_functions': ['get_input_schema', 'get_input_jsonschema', 'get_output_schema', 'get_output_jsonschema', 'get_subgraphs', 'aget_subgraphs'], 'summary': 'This file is a critical component of the LangGraph framework, responsible for managing graph data and subgraphs. However, its high opacity score indicates significant structural issues. The file assumes specific keys exist in config dicts

In [7]:
#@title 1.6 Run Mistral 7B
print("--- Mistral 7B ---")
start_server(MISTRAL_7B_FILE, n_gpu_layers=-1, wait=30)
result_mistral_7b = run_prompt(OPACITY_SYSTEM, "Mistral 7B opacity")
burden_mistral_7b = run_prompt(CALLER_BURDEN_SYSTEM, "Mistral 7B caller burden", 600)
print_result(result_mistral_7b, "Mistral 7B")

--- Mistral 7B ---
  Loading Mistral-7B-Instruct-v0.3-Q5_K_M.gguf (30s)...
  Server ready
  Running Mistral 7B opacity...
{'undeclared_preconditions': 12, 'global_state_risk': 8, 'implicit_coupling': 10, 'contract_completeness': 12, 'structural_centrality': 18, 'total': 60, 'opacity_score': 0.33, 'primary_signal': 'This file is part of a complex AI agent framework and has a high level of implicit assumptions and global state usage.', 'load_bearing_functions': ['get_config_jsonschema', 'get_context_jsonschema', 'get_input_schema', 'get_input_jsonschema', 'get_output_schema', 'get_output_jsonschema', 'get_subgraphs', 'aget_subgraphs'], 'summary': 'This Python file is a core component of the LangGraph AI agent framework, responsible for managing Pregel graph operations. It has a high level of complexity and relies on various global states and implicit assumptions.'}
  Running Mistral 7B caller burden...
{'undeclared_preconditions': 0, 'global_state_risk': 0, 'implicit_coupling': 0, 'contr

In [8]:
#@title 1.7 Ensemble comparison: Apertus 8B vs Mistral 7B
print_comparison(result_apertus_8b, result_mistral_7b,
                 burden_apertus_8b, burden_mistral_7b,
                 "Apertus 8B", "Mistral 7B")


 ENSEMBLE COMPARISON: Apertus 8B vs Mistral 7B
  Apertus 8B             [XXXXXXXXXXXXXXXXXXXXXXXX........]  78/100
  Mistral 7B             [XXXXXXXXXXXXXXXXXXX.............]  60/100
  Spread: 18 pts  Overall: DIVERGE

------------------------------------------------------------------------  DIMENSIONS
  Dimension                  Apertus 8B  Mistral 7B  Delta  Signal
  Undeclared preconditions           12          12      0  CONVERGE
  Global state risk                  16           8     -8  CONVERGE
  Implicit coupling                  14          10     -4  CONVERGE
  Contract completeness              12          12      0  CONVERGE
  Structural centrality              18          18      0  CONVERGE

------------------------------------------------------------------------  PRIMARY SIGNAL
  Apertus 8B: This file has a high opacity score due to its reliance on global state, implicit coupling, and incomplete contracts. The file assumes specific keys exist in config dicts, relies o

---
## Section 2 — Depth of Read: Apertus 70B vs Mistral 7B
**Requires A100 — any paid Colab plan.**

Replaces Apertus 8B with 70B. Tests whether a much larger model finds structural
contracts that 8B-class models read past. This is the Appendix A reproduction.

Standard A100 (40GB): set `A100_70B_PARTIAL_LAYERS` in Section 0 if needed,
or enable High-RAM via Runtime > Change runtime type (Pro or Pro+).

In [9]:
#@title 2.1 Download Apertus 70B
if not can_run_s2:
    print("Section 2 requires A100 (any paid Colab plan).")
    print("Section 1 above fully demonstrates the methodology.")
else:
    quant = "5_K_S" if (is_high_ram and is_a100) else "4_K_M"
    APERTUS_70B_FILE = f"Apertus-70B-Instruct-2509-Q{quant}.gguf"
    APERTUS_70B_REPO = "unsloth/Apertus-70B-Instruct-2509-GGUF"
    print(f"Downloading Apertus 70B Q{quant}  (5-15 min)...")
    if not os.path.exists(f"/content/{APERTUS_70B_FILE}"):
        hf_hub_download(repo_id=APERTUS_70B_REPO,
                        filename=APERTUS_70B_FILE, local_dir="/content")
    print(f"  {APERTUS_70B_FILE}")

Apertus-70B-Instruct-2509-Q5_K_S.gguf:   0%|          | 0.00/48.7G [00:00<?, ?B/s]

  Apertus-70B-Instruct-2509-Q5_K_S.gguf


In [10]:
#@title 2.2 Run Apertus 70B
if not can_run_s2:
    print("Skipping.")
else:
    print(f"--- Apertus 70B (layers: {layers_70b}) ---")
    start_server(APERTUS_70B_FILE, n_gpu_layers=layers_70b, wait=60)
    result_apertus_70b = run_prompt(OPACITY_SYSTEM, "Apertus 70B opacity")
    burden_apertus_70b = run_prompt(CALLER_BURDEN_SYSTEM, "Apertus 70B caller burden", 600)
    print_result(result_apertus_70b, "Apertus 70B")

--- Apertus 70B (layers: -1) ---
  Loading Apertus-70B-Instruct-2509-Q5_K_S.gguf (60s)...
  Server ready
  Running Apertus 70B opacity...
{'undeclared_preconditions': 12, 'global_state_risk': 14, 'implicit_coupling': 15, 'contract_completeness': 10, 'structural_centrality': 18, 'total': 61, 'opacity_score': 0.344, 'primary_signal': "The `get_subgraphs` method assumes callers know the graph's namespace structure and that subgraphs are accessible via `node.subgraphs`.", 'load_bearing_functions': ['aget_subgraphs', 'get_subgraphs', 'get_input_schema', 'get_output_schema', 'get_input_jsonschema', 'get_output_jsonschema'], 'summary': 'This file contains core Pregel algorithm logic and graph manipulation utilities. It has high structural centrality due to its role in graph processing and subgraph traversal. It assumes callers understand graph structure and namespace conventions, and reads/writes global state via `self.channels` and `self.config`. It also implicitly relies on external configu

In [11]:
#@title 2.3 Comparison: Apertus 70B vs Mistral 7B  (Appendix A reproduction)
if not can_run_s2:
    print("Skipping.")
else:
    print_comparison(result_apertus_70b, result_mistral_7b,
                     burden_apertus_70b, burden_mistral_7b,
                     "Apertus 70B", "Mistral 7B")
    # Appendix A check
    items_70 = burden_apertus_70b.get("caller_must_know",[])
    items_7  = burden_mistral_7b.get("caller_must_know",[])
    stated_70 = all(i.get("stated_in_interface",False) for i in items_70)
    stated_7  = all(i.get("stated_in_interface",False) for i in items_7)
    print()
    print("APPENDIX A CHECK:")
    print(f"  Apertus 70B caller burden: {'ALL STATED' if stated_70 else 'UNSTATED items found'}")
    print(f"  Mistral 7B  caller burden: {'ALL STATED' if stated_7  else 'UNSTATED items found'}")
    if not stated_70:
        print("  70B found deeper orphaned invariants — paper finding confirmed")


 ENSEMBLE COMPARISON: Apertus 70B vs Mistral 7B
  Apertus 70B            [XXXXXXXXXXXXXXXXXXX.............]  61/100
  Mistral 7B             [XXXXXXXXXXXXXXXXXXX.............]  60/100
  Spread: 1 pts  Overall: CONVERGE

------------------------------------------------------------------------  DIMENSIONS
  Dimension                  Apertus 70  Mistral 7B  Delta  Signal
  Undeclared preconditions           12          12      0  CONVERGE
  Global state risk                  14           8     -6  CONVERGE
  Implicit coupling                  15          10     -5  CONVERGE
  Contract completeness              10          12     +2  CONVERGE
  Structural centrality              18          18      0  CONVERGE

------------------------------------------------------------------------  PRIMARY SIGNAL
  Apertus 70B: The `get_subgraphs` method assumes callers know the graph's namespace structure and that subgraphs are accessible via `node.subgraphs`.
  Mistral 7B: This file is part of a comp

---
## Section 3 — Full Architecture Differential: Apertus 70B vs Mistral Nemo 12B
**Requires A100 — any paid Colab plan.**

The most heterogeneous comparison in the notebook: Apertus 70B (Swiss, Llama-family)
vs Mistral Nemo 12B (French, independent architecture, different training corpus).
A finding confirmed at this diversity level approaches the paper's 3x confidence
threshold for confirmed orphaned invariants.


In [12]:
#@title 3.1 Download Mistral Nemo 12B
if not can_run_s2:
    print("Section 3 requires A100.")
    print()
    print("What Section 3 adds:")
    print("  Mistral Nemo 12B is from a completely different training lineage")
    print("  than Apertus. Convergence between Apertus 70B and Mistral Nemo")
    print("  tests whether structural signals cross the Swiss/French architectural")
    print("  boundary — not just the 8B/70B size boundary.")
    print("  Convergence here approaches the paper's 3x confirmation threshold.")
else:
    NEMO_FILE = "Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"
    NEMO_REPO = "bartowski/Mistral-Nemo-Instruct-2407-GGUF"
    print(f"Downloading Mistral Nemo 12B...")
    if not os.path.exists(f"/content/{NEMO_FILE}"):
        hf_hub_download(repo_id=NEMO_REPO, filename=NEMO_FILE, local_dir="/content")
    print(f"  {NEMO_FILE}")

Mistral-Nemo-Instruct-2407-Q5_K_M.gguf:   0%|          | 0.00/8.73G [00:00<?, ?B/s]

  Mistral-Nemo-Instruct-2407-Q5_K_M.gguf


In [13]:
#@title 3.2 Run Mistral Nemo 12B
if not can_run_s2:
    print("Skipping.")
else:
    print("--- Mistral Nemo 12B ---")
    start_server(NEMO_FILE, n_gpu_layers=-1, wait=45)
    result_nemo = run_prompt(OPACITY_SYSTEM, "Mistral Nemo 12B opacity")
    burden_nemo = run_prompt(CALLER_BURDEN_SYSTEM, "Mistral Nemo 12B caller burden", 600)
    print_result(result_nemo, "Mistral Nemo 12B")

--- Mistral Nemo 12B ---
  Loading Mistral-Nemo-Instruct-2407-Q5_K_M.gguf (45s)...
  Server ready
  Running Mistral Nemo 12B opacity...
{'undeclared_preconditions': 12, 'global_state_risk': 15, 'implicit_coupling': 10, 'contract_completeness': 10, 'structural_centrality': 18, 'total': 65, 'opacity_score': 0.65, 'primary_signal': 'High opacity due to global state usage and implicit coupling.', 'load_bearing_functions': ['run', 'get_subgraphs', 'aget_subgraphs'], 'summary': "The file 'main.py' in the LangGraph repository exhibits high opacity due to its reliance on global state, implicit coupling, and central role in the system. It assumes certain external resources and orderings, and its correctness depends on other parts of the system."}
  Running Mistral Nemo 12B caller burden...
  [parse_json] JSONDecodeError: Unterminated string starting at: line 1 column 1353 (char 1352)
  [parse_json] Raw (first 200 chars): {"undeclared_preconditions": 1, "global_state_risk": 2, "implicit_coupling

In [14]:
#@title 3.3 Comparison: Apertus 70B vs Mistral Nemo 12B
if not can_run_s2:
    print("Skipping.")
else:
    print_comparison(result_apertus_70b, result_nemo,
                     burden_apertus_70b, burden_nemo,
                     "Apertus 70B", "Mistral Nemo 12B")


 ENSEMBLE COMPARISON: Apertus 70B vs Mistral Nemo 12B
  Apertus 70B            [XXXXXXXXXXXXXXXXXXX.............]  61/100
  Mistral Nemo 12B       [XXXXXXXXXXXXXXXXXXXX............]  65/100
  Spread: 4 pts  Overall: CONVERGE

------------------------------------------------------------------------  DIMENSIONS
  Dimension                  Apertus 70  Mistral Ne  Delta  Signal
  Undeclared preconditions           12          12      0  CONVERGE
  Global state risk                  14          15     +1  CONVERGE
  Implicit coupling                  15          10     -5  CONVERGE
  Contract completeness              10          10      0  CONVERGE
  Structural centrality              18          18      0  CONVERGE

------------------------------------------------------------------------  PRIMARY SIGNAL
  Apertus 70B: The `get_subgraphs` method assumes callers know the graph's namespace structure and that subgraphs are accessible via `node.subgraphs`.
  Mistral Nemo 12B: High opacity du

In [15]:
#@title 3.4 Four-model convergence summary (runs if all sections completed)
from collections import Counter

available = {"Apertus 8B": result_apertus_8b,
             "Mistral 7B":  result_mistral_7b}
if can_run_s2:
    available["Apertus 70B"] = result_apertus_70b
if can_run_s2:
    available["Mistral Nemo 12B"] = result_nemo

W = 72
print(f"{'='*W}")
print(f" CROSS-MODEL CONVERGENCE SUMMARY  ({len(available)} models)")
print(f"{'='*W}")

print("\n  Primary signals:")
sg_count = 0
for lbl, res in available.items():
    ps = res.get("primary_signal","")
    found = "subgraph" in ps.lower()
    if found: sg_count += 1
    print(f"  {'[MATCH]' if found else '[      ]'} {lbl:<22} {ps[:50]}")

print(f"\n  get_subgraphs named by {sg_count}/{len(available)} models")
if sg_count >= 3:
    print(f"  Confirmed orphaned invariant at {sg_count}x confidence")
    print("  Meets or exceeds the paper's 3x threshold")
elif sg_count == 2:
    print("  Confirmed at 2x — run more sections for 3x threshold")

print("\n  Load-bearing function counts across all models:")
fn_counter = Counter()
for res in available.values():
    for fn in res.get("load_bearing_functions",[]):
        fn_counter[fn.lower()] += 1
for fn, count in fn_counter.most_common(8):
    bar = "X" * count
    print(f"  {bar:<6} {fn:<35} [{count}x]")

print(f"\n{'='*W}")
n = len(available)
lineages = []
if "Apertus 8B" in available or "Apertus 70B" in available:
    lineages.append("Swiss (CSCS / Llama-family)")
if "Mistral 7B" in available or "Mistral Nemo 12B" in available:
    lineages.append("French (Mistral architecture)")
print(f"  {n} model{'s' if n>1 else ''}. {len(lineages)} training {'lineages' if len(lineages)>1 else 'lineage'}.")
if len(lineages) > 1:
    print(f"  {' vs '.join(lineages)}.")
if sg_count >= 2:
    print("  The orphaned invariant in get_subgraphs is not a model artifact.")
    print("  It is a structural property of the code.")
print(f"{'='*W}")

 CROSS-MODEL CONVERGENCE SUMMARY  (4 models)

  Primary signals:
  [      ] Apertus 8B             This file has a high opacity score due to its reli
  [      ] Mistral 7B             This file is part of a complex AI agent framework 
  [MATCH] Apertus 70B            The `get_subgraphs` method assumes callers know th
  [      ] Mistral Nemo 12B       High opacity due to global state usage and implici

  get_subgraphs named by 1/4 models

  Load-bearing function counts across all models:
  XXXX   get_subgraphs                       [4x]
  XXXX   aget_subgraphs                      [4x]
  XXX    get_input_schema                    [3x]
  XXX    get_input_jsonschema                [3x]
  XXX    get_output_schema                   [3x]
  XXX    get_output_jsonschema               [3x]
  X      get_config_jsonschema               [1x]
  X      get_context_jsonschema              [1x]

  4 models. 2 training lineages.
  Swiss (CSCS / Llama-family) vs French (Mistral architecture).
